In [1]:
import torch
import torch.nn.functional as F


In [2]:
def generate_masked_attn(a, b, value_mask = -1):
    a = a.detach().clone()

    for i in range(b.size()[0]):
        a[i, b[:i]] = -1
    # end
    return a
# end

In [3]:
unmask = torch.load('stats/0/unmask_96_160.pt')
unmask = unmask.squeeze(-1) - 32

In [4]:
attn = torch.load('stats/0/attn_32_96.pt')
attn_last = attn[:,-1,:,:].squeeze(1) # -> [step, attn]
attn_last_switched = attn_last[unmask, :]
# attn_lst_switched_masked = generate_masked_attn(attn_last_switched, unmask)

In [5]:
# evaluation the sequence switch
# attn_mocked = torch.arange(64).unsqueeze(1).repeat(1,64).to('cuda:0')
# attn_mocked[unmask, :]

/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [16,0,0], thread: [0,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [16,0,0], thread: [1,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [16,0,0], thread: [2,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [16,0,0], thread: [3,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [16,0,0], thread: [4,0,0] Assertion 

In [ ]:
T, L = 64,64
attn = attn_last_switched_masked
order = unmask

dev = attn.device

step_of = torch.full((L,), -1, dtype=torch.long, device=dev)   # [L] step at which a position is unmasked, -1 = never
step_of[order] = torch.arange(T, device=dev)                   # step_of 是指这个token在第几步被unmask的

t_idx = torch.arange(T, device=dev).view(T, 1)
future_gap = step_of.view(1, L) - t_idx                        # [T, L]  gap>0 -> future candidate, gap=1 -> next
cand_mask = future_gap > 0                                     # [T, L]  still-masked at step t

neg_inf = torch.finfo(attn.dtype).min
S = attn.masked_fill(~cand_mask, neg_inf)                 # [T, L]  attention restricted to future candidates

n_cand = (T - 1) - torch.arange(T, device=dev)                 # [T]  number of still-masked tokens after step t


NameError: name 'attn_lst_switched_masked' is not defined